# 🧠 Growth Intelligence Agent
## A Sequential Multi-Agent AI System for Product & Market Intelligence

> **One question in. A complete intelligence report out.**

---

You ask: *"What's the growth potential for an AI-powered CRM tool?"*

The system deploys **6 specialized AI agents** in sequence, each analysing a different dimension of the market — and delivers a fully structured, source-attributed **PDF intelligence report** with an executive summary.

---
# 🚨 The Problem We Solve

Traditional market research is:
- **Slow** — weeks to gather analyst reports, competitor data, pricing intelligence
- **Fragmented** — market trends live in one place, competitor pricing in another, customer sentiment in Reddit
- **Expensive** — Gartner/Forrester subscriptions, consultants, research teams
- **Stale** — by the time a report is ready, the market has moved

### Our Solution
A fully automated, AI-driven intelligence pipeline that:
1. Searches the live web in real time
2. Scrapes competitor pages, pricing tables, landing copy
3. Reads Reddit and Hacker News for raw customer sentiment
4. Structures all raw data into a rigorous, source-attributed intelligence report
5. Delivers a downloadable PDF — in minutes, not weeks

---
# 🏗️ System Architecture Overview

The system is built on **LangGraph** — a state machine for AI workflows — and uses a sequential multi-agent pattern.

```
┌─────────────────────────────────────────────────────────────────────┐
│                        CHAT INTERFACE (Next.js)                     │
│                     agent-chat-ui (useStream hook)                  │
└──────────────────────────────┬──────────────────────────────────────┘
                               │  User question (HumanMessage)
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    LANGGRAPH ORCHESTRATOR                           │
│                                                                     │
│  [receive_message] → [classify_domains] → [domain agents] →        │
│  [synthesise] → [export_pdf] → [stream_response] → [update_memory] │
│                                                                     │
│              Shared State: OrchestratorState (Pydantic)            │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
│  DOMAIN AGENTS   │  │  MICRO-AGENTS    │  │  MEMORY LAYER    │
│                  │  │                  │  │                  │
│  Market          │  │  Search          │  │  Mem0            │
│  Competitive     │  │  (Gemini Flash)  │  │  (thread memory) │
│  Win/Loss        │  │                  │  │                  │
│  Pricing         │  │  Scrape          │  │  Pinecone        │
│  Positioning     │  │  (Firecrawl)     │  │  (vector store)  │
│  Adjacent        │  │                  │  │                  │
└──────────────────┘  │  Social          │  └──────────────────┘
                       │  (Reddit + HN)   │
                       │                  │
                       │  Parse           │
                       │  (LlamaParse)    │
                       │                  │
                       │  Synthesise      │
                       │  (Claude Sonnet) │
                       └──────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                        PDF EXPORT                                   │
│              WeasyPrint + Jinja2 → base64 data URL                 │
│                  Delivered directly in chat response                │
└─────────────────────────────────────────────────────────────────────┘
```

---
# 🏗️ System Architecture — Mermaid Diagram

```mermaid
flowchart TD
    User(["👤 User"]) -->|Question| UI["Chat Interface\nNext.js · agent-chat-ui"]

    UI -->|HumanMessage| recv

    subgraph ORCH["⚙️ LangGraph StateGraph"]
        recv["receive_message\n· extract query · reset turn state\n· load Mem0 context"]
        recv --> classify["classify_domains\nClaude Sonnet 4.5\nselects 1–6 domains"]
        classify --> router{{"Domain Router\nconditional edges"}}

        router --> mkt["run_market"]
        router --> cmp["run_competitive"]
        router --> wl["run_win_loss"]
        router --> pri["run_pricing"]
        router --> pos["run_positioning"]
        router --> adj["run_adjacent"]

        mkt & cmp & wl & pri & pos & adj --> synth["synthesise\nClaude Sonnet 4.5\nexecutive summary"]
        synth --> epdf["export_pdf\nWeasyPrint + Jinja2\nbase64 data URL"]
        epdf --> stream["stream_response\nsummary + PDF link"]
        stream --> updmem["update_memory\nwrite to Mem0"]
    end

    subgraph PIPELINE["🤖 Domain Agent Pipeline — runs for each of the 6 agents"]
        direction LR
        p1["① Check\nMem0 cache"] --> p2["② Form\nsub-queries"] --> p3["③ Gather\nsignals"] --> p4["④ Chunk\n512 tokens"] --> p5["⑤ Embed +\nupsert"] --> p6["⑥ Recall\ntop-5"] --> p7["⑦ Claude\nsynthesis"]
    end

    subgraph MICRO["🔧 Micro-Agents"]
        srch["🔍 Search\nGemini Flash 2.0\nGoogle Grounding"]
        scrp["🕷️ Scrape\nFirecrawl\nHTML → Markdown"]
        soc["💬 Social\nReddit asyncpraw\n+ HN Algolia"]
        pars["📄 Parse\nLlamaParse\nPDF → Markdown"]
    end

    subgraph MEM["🗄️ Memory Layer"]
        mem0[("Mem0\nThread Memory\ncross-session")]
        pine[("Pinecone\nVector Store\n3072-dim cosine")]
    end

    mkt & cmp & wl & pri & pos & adj -.->|"7-step pipeline"| PIPELINE
    p3 --> srch & scrp & soc
    scrp -->|"PDF detected"| pars
    p5 --> pine
    p6 <-->|"semantic search"| pine
    p1 <-->|"get_prior()"| mem0
    updmem -->|"set_global_context()"| mem0

    stream -->|"Answer + download link"| UI
    epdf --> PDF[/"📄 Intelligence Report PDF"/]
```


---
# 🔄 End-to-End Data Flow

### Step-by-step: what happens when you ask a question

```
USER: "What is the growth potential for an AI-powered CRM?"
         │
         ▼
① receive_message
   • Extracts the HumanMessage from the chat stream
   • Creates a unique session_id (UUID)
   • Loads prior session context from Mem0 (if exists)
   • Writes: { session_id, user_query } → OrchestratorState
         │
         ▼
② classify_domains  [Claude Sonnet 4.5]
   • Reads the user_query
   • Asks Claude: "Which of [market, competitive, win_loss,
     pricing, positioning, adjacent] are relevant?"
   • Claude returns: ["market", "competitive", "pricing"]
   • Writes: { relevant_domains } → OrchestratorState
         │
         ▼
③ run_market  [MarketAgent]
④ run_competitive  [CompetitiveAgent]           ← Sequential
⑤ run_pricing  [PricingAgent]                   ← each domain
   Each agent runs the same 7-step pipeline:         runs after
   1. Check Mem0 (already answered this?)             the previous
   2. Form 3-4 domain-specific sub-queries            completes
   3. Search → Scrape → Social (signal gathering)
   4. Chunk all content into 512-token pieces
   5. Embed + upsert to Pinecone (vector DB)
   6. Recall top-5 most relevant chunks
   7. Claude Sonnet synthesises → DomainFinding
   8. Save finding to Mem0
         │
         ▼
⑥ synthesise  [Claude Sonnet 4.5]
   • Reads all DomainFindings
   • Writes 3-5 sentence executive summary
   • Cross-domain strategic recommendations
         │
         ▼
⑦ export_pdf  [WeasyPrint + Jinja2]
   • Renders FinalReport → HTML template
   • Converts HTML → PDF
   • Encodes PDF as base64 data URL
         │
         ▼
⑧ stream_response
   • Sends executive summary + download link to chat UI
         │
         ▼
⑨ update_memory  [Mem0]
   • Stores query, domains run, summary in global context
   • Available for next conversation in this session
```

---
# 📋 State Management — The Shared Whiteboard

## OrchestratorState (Pydantic Model)

Every node in the LangGraph reads from and writes to a **single shared state object** called `OrchestratorState`. Think of it as a whiteboard that every agent can see and write on.

```python
class OrchestratorState(BaseModel):
    messages:         list[BaseMessage]          # Chat history (auto-appended)
    session_id:       str                        # Unique conversation ID
    user_query:       str                        # Original question
    relevant_domains: list[str]                  # Domains chosen by classifier
    domain_findings:  dict[str, DomainFinding]  # Results from each agent
    error_domains:    list[str]                  # Agents that failed
    pdf_url:          str | None                 # Final download URL
```

### How State Flows Through the System

| Node | Reads From State | Writes To State |
|------|------------------|-----------------|
| `receive_message` | `messages` (latest) | `session_id`, `user_query` |
| `classify_domains` | `user_query` | `relevant_domains` |
| `run_market` | `user_query`, `session_id` | `domain_findings["market"]` |
| `run_competitive` | `user_query`, `session_id` | `domain_findings["competitive"]` |
| `synthesise` | `domain_findings` (all) | `domain_findings["synthesis"]` |
| `export_pdf` | `domain_findings`, `session_id` | `pdf_url` |
| `stream_response` | `domain_findings["synthesis"]`, `pdf_url` | `messages` (final answer) |
| `update_memory` | full state | (writes to Mem0, not state) |

### Key Design Decision — Immutable Updates
LangGraph nodes never mutate state directly. Each node returns a **dict of changes**, and LangGraph merges them:

```python
# Each agent returns a patch, not a mutated state
return {
    "domain_findings": {**state.domain_findings, "market": finding},
    "messages": [progress_msg],  # Appended via add_messages reducer
}
```

The `messages` field uses LangChain's `add_messages` reducer — meaning messages are always **appended**, never replaced. This is what makes the chat history grow naturally.

---
# 🔀 Dynamic Routing — Not All 6 Agents Always Run

## The Domain Classifier

One of our **novel design choices**: Claude decides at runtime which subset of agents to run, based on the user's question.

```
Question: "How should I price my SaaS product?"
→ Claude routes to: [pricing, competitive, positioning]
→ market, win_loss, adjacent are SKIPPED
→ Faster response, lower cost, more focused output

Question: "Should I build a Salesforce competitor?"
→ Claude routes to: [market, competitive, win_loss, pricing, positioning, adjacent]
→ Full 6-agent run
→ Complete 360° intelligence report
```

## Conditional Graph Edges

The graph uses **conditional routing** — after each domain completes, the router checks which domain is next in the relevant list:

```python
def _next_relevant_domain(relevant_domains, after="market"):
    # Walk the ordered list [market, competitive, win_loss, pricing, positioning, adjacent]
    # Return the next one that is in relevant_domains
    # Return None if none remain → routes to synthesise
```

Fixed execution order: `market → competitive → win_loss → pricing → positioning → adjacent`
But **any of them can be skipped** based on the classifier's decision.

## Graceful Failure Isolation

Each domain node is wrapped in `try/except`. If one agent fails:
- A `PartialFinding(status="failed")` is stored instead
- The domain is added to `error_domains`
- The pipeline continues to the next domain
- The synthesis notes which domains failed
- **The whole system never crashes** — it degrades gracefully

---
# 🤖 The Six Domain Agents

Each agent inherits from `BaseDomainAgent` and implements the same 7-step pipeline. The differences are **what they search for**, **which sources they emphasise**, and **how they structure their findings**.

---

## 1️⃣ Market Agent — *"Where is the category heading?"*

**Core Question**: Is this market growing? What are the leading indicators? How big is the opportunity?

**Sub-Queries Generated**:
```
• "{query} market size trends 2026"
• "{query} industry growth forecast leading indicators"
• "{query} venture capital funding investment trends"
• "{query} patent filings innovation trends 2025 2026"  ← unique to Market agent
```

**Signal Sources**: Gartner/Forrester reports, VC announcements, USPTO patents, Google Trends, Reddit discussions

**Finds**: CAGR, market size ($), number of new entrants, hype cycle position, VC round velocity

**PDF Section**: Market & Trends — growth projections, investment signals, leading indicators

---

## 2️⃣ Competitive Agent — *"Who is doing what? Can this bet be won?"*

**Core Question**: Who are the main players? How are they funded, positioned, and priced?

**Sub-Queries Generated**:
```
• "{query} top competitors comparison 2026"
• "{query} competitor pricing features"
• "{query} competitive landscape analysis"
• "{query} G2 Capterra reviews competitor comparison"   ← unique to Competitive
• "{query} funding rounds investors Series A B 2025 2026" ← unique to Competitive
```

**Signal Sources**: Competitor landing pages (scraped), G2 reviews, Meta Ad Library, funding databases

**Finds**: Competitor list, pricing tiers, G2 scores, Series A/B rounds, feature matrices, market share

**PDF Section**: Competitive Landscape — scorecard, positioning map, funding context

---

## 3️⃣ Win/Loss Agent — *"Why do buyers choose this? Why do they leave?"*

**Core Question**: What drives purchase? What are the top churn reasons from the buyer's perspective?

**Sub-Queries Generated**:
```
• "{query} customer reviews complaints why switched"
• "{query} win loss analysis buyer reasons"
• "{query} product failures churn reasons Reddit"
• "{query} customer success stories case studies"        ← unique to Win/Loss
• "{query} churn reasons buyer feedback site:reddit.com" ← unique to Win/Loss
```

**Signal Sources**: Reddit (r/sales, r/startups), Hacker News "Ask HN" threads, G2 reviews, case study blogs — **social is most heavily weighted here**

**Finds**: Win reasons (human-quality AI, fast setup), loss reasons (robotic emails, CRM failures, unclear ROI at 6 months), onboarding friction

**PDF Section**: Win/Loss Analysis — churn drivers, conversion blockers, verbatim customer quotes

---

## 4️⃣ Pricing Agent — *"Is the pricing model right? What will buyers pay?"*

**Core Question**: What pricing models work? What is WTP (willingness to pay) anchored to?

**Sub-Queries Generated**:
```
• "{query} pricing model comparison willingness to pay"
• "{query} pricing strategy SaaS benchmark"
• "{query} pricing page analysis competitors"
• "{query} pricing page tiers monthly annual plans"      ← unique to Pricing
• "site:reddit.com {query} pricing too expensive worth it" ← unique to Pricing
```

**Signal Sources**: Competitor pricing pages (Firecrawl scrape), SaaS pricing reports, Reddit pricing threads

**Finds**: Price ranges, seat vs. usage vs. outcome-based models, annual vs monthly churn difference, WTP anchors, freemium conversion data

**PDF Section**: Pricing & Packaging — pricing matrix, model comparison, WTP analysis, recommended model

---

## 5️⃣ Positioning Agent — *"How should this be talked about? What message is unowned?"*

**Core Question**: What are competitors claiming? What messaging gap exists to own?

**Sub-Queries Generated**:
```
• "{query} messaging positioning unique value proposition"
• "{query} marketing copy differentiation analysis"
• "{query} brand positioning competitor messaging gap"
• "{query} homepage value proposition tagline messaging"  ← unique to Positioning
• "{query} marketing copy ads creative 2026"              ← unique to Positioning
```

**Signal Sources**: Landing pages (Firecrawl), Meta Ad Library ads, SEO content — ⚠️ **social is intentionally SKIPPED** (Reddit doesn't reflect messaging strategy)

**Finds**: Saturated positioning claims ("10x pipeline"), unowned messaging gaps ("security-first", "compliance-centric"), thought leadership vacuums, SEO content opportunities

**PDF Section**: Positioning & Messaging — messaging map, gap analysis, competitor copy samples, recommended positioning

---

## 6️⃣ Adjacent Agent — *"What's coming from outside? What's the disruption risk?"*

**Core Question**: What big tech is entering? What adjacent category will collide with this one?

**Sub-Queries Generated**:
```
• "{query} adjacent market disruption threat 2026"
• "{query} platform encroachment substitute products"
• "{query} emerging competitors adjacent category"
• "{query} platform entering market disruption big tech"  ← unique to Adjacent
• "{query} adjacent market patents 2025 2026 new entrant" ← unique to Adjacent
```

**Signal Sources**: Hacker News (rich for platform threat signals), USPTO patents, SerpAPI news, big tech product announcements

**Finds**: Platform bundling threats (Salesforce/HubSpot/Microsoft entering), patent activity by incumbents, timeline to commoditisation

**PDF Section**: Adjacent Market Threats — threat timeline, consolidation risks, strategic implications

---
# 🔧 Micro-Agents — The Signal Pipeline

Every domain agent calls the same 3 micro-agents to gather raw signals. These are the "workers" — they don't reason, they just fetch and normalise data.

---

## 🔍 Search Micro-Agent
**Technology**: Gemini Flash 2.5 with Google Search grounding

**What it does**: Runs each sub-query through Gemini, which uses Google's live search as a built-in tool. Returns structured `SearchResult` objects with source URLs and grounding metadata.

**Input**: `"AI CRM market size trends 2026"`
**Output**: `[SearchResult(url="statista.com/...", title="CRM Market Size", snippet="...", raw_content="...")]`

**Novel detail**: Gemini's grounding provides **source attribution out of the box** — every result has a verified source URL from Google's index.

---

## 🕷️ Scrape Micro-Agent
**Technology**: Firecrawl (HTML → Markdown) + LlamaParse (PDF deep-parse)

**What it does**: Takes URLs from search results, converts full web pages to clean markdown. If a URL points to a PDF (research report, whitepaper), it routes automatically to LlamaParse for OCR + layout-aware extraction.

**Input**: `"https://gartner.com/crm-report-2026.pdf"`
**Output**: `ScrapeResult(markdown="# CRM Market Report 2026\n\n...full clean text", is_pdf=False)`

**Novel detail**: **Automatic PDF routing** — when Firecrawl detects `content_type: application/pdf`, it flags `is_pdf=True` and the caller silently reroutes to LlamaParse. No manual intervention needed.

---

## 💬 Social Micro-Agent
**Technology**: Reddit OAuth2 (asyncpraw) + Hacker News Algolia public API

**What it does**: Searches Reddit across all subreddits and pulls the top 15 posts with their top-5 comments. In parallel, hits HN's Algolia search API for story headlines and text.

**Input**: `"AI CRM customer reviews complaints"`
**Output**: `[Post(title="Why we left Salesforce...", body="...", comments=["We switched because..."], source="reddit")]`

**Novel detail**: **Comments are captured**, not just post titles. Each Reddit post pulls up to 5 top-level comments — this is where the real signal lives ("The integration was broken", "Too expensive for the ROI").

---

## 📄 Parse Micro-Agent
**Technology**: LlamaParse

**What it does**: Downloads a PDF, writes to a temp file, and uses LlamaParse's OCR + layout intelligence to extract clean markdown. Handles complex documents like tables, multi-column layouts, and charts.

**Input**: PDF URL
**Output**: `ParseResult(content="full parsed markdown including tables")`

---

## 🧠 Synthesis Micro-Agent
**Technology**: Claude Sonnet 4.5 + Pydantic validation + few-shot retry loop

**What it does**: Takes the top-5 most relevant chunks and synthesises them into a structured `DomainFinding` JSON object. Uses a **few-shot prompt** with 3 domain-specific examples to show Claude exactly what format to produce.

**Novel detail — retry loop**: If Claude returns malformed JSON, the system sends the error back to Claude:
```
Attempt 1: Claude returns { confidence: 1.5 }  ← invalid (max is 1.0)
System: "Your response failed Pydantic validation: confidence must be ≤ 1.0. Fix it."
Attempt 2: Claude returns { confidence: 0.87 }  ← valid ✓
```
Up to 3 retry attempts before failing with `SynthesisFailedError`.

---
# 🗄️ Memory Architecture — Two Layers

The system has **two completely separate memory systems** serving different purposes:

```
┌────────────────────────────────────┐  ┌────────────────────────────────────┐
│          MEM0 (Thread Memory)      │  │       PINECONE (Vector Store)       │
│                                    │  │                                    │
│  "What did we find last time?"     │  │  "What's most relevant to this     │
│                                    │  │    specific question RIGHT NOW?"   │
│  Semantic deduplication:           │  │                                    │
│  If you ask the same Q again,      │  │  Stores 512-token chunks           │
│  Mem0 deduplicates & updates       │  │  Embedded with OpenAI              │
│  in-place — no duplicates          │  │  text-embedding-3-large            │
│                                    │  │  (3072 dimensions)                 │
│  Persists ACROSS sessions          │  │                                    │
│                                    │  │  Namespaced per session_id         │
│  Two scopes:                       │  │  Tagged per domain_tag             │
│  • global:{session_id}             │  │                                    │
│    (session-wide context)          │  │  top_k = 5 (always)                │
│  • domain:{session_id}:{domain}    │  │  Cosine similarity search          │
│    (per-agent finding history)     │  │                                    │
└────────────────────────────────────┘  └────────────────────────────────────┘
```

---

## Mem0 — Thread Memory ("What did we find before?")

Mem0 acts like a smart notebook that **updates intelligently** instead of duplicating.

**Two scopes**:

1. **Global thread** (`global:{session_id}`)
   - Stores the overall session context: what question was asked, which domains were run, the executive summary
   - Loaded at the start of every conversation
   - Lets the system remember *"Last time you asked about CRM, we found X"*

2. **Domain threads** (`domain:{session_id}:{domain}`)
   - Each domain agent stores its `DomainFinding` here after completing
   - On the next run, the agent calls `get_prior()` first
   - If `is_already_answered()` returns True → **skip the entire pipeline**, return cached result
   - This avoids redundant API calls on repeat questions

**How Mem0's deduplication works**:
- When you write to Mem0, it doesn't just append a new record
- It semantically matches the new content against existing memories
- If a similar memory exists, it **updates in-place** rather than creating a duplicate
- This keeps memory lean and prevents stale data accumulation

---

## Pinecone — Vector Store ("What's most relevant to THIS question?")

Pinecone stores every piece of scraped content as a dense vector for semantic search.

**The chunking pipeline**:
```
Raw text (from search + scrape + social)
    ↓
RecursiveCharacterTextSplitter
    chunk_size = 512 tokens (~2000 chars)
    chunk_overlap = 64 tokens (~256 chars)
    ↓
Each chunk gets:
    id: UUID4
    text: 512-token excerpt
    source_url: where it came from
    domain_tag: "market" / "competitive" / etc.
    agent_id: "MarketAgent" / etc.
    ↓
OpenAI text-embedding-3-large
    3072-dimensional dense vector
    ↓
Upserted to Pinecone
    Namespace: session_id (isolated per conversation)
    Batched: 100 vectors per API call
```

**Retrieval**:
- Query: embed the user's question
- Filter by `domain_tag` (so market chunks don't pollute competitive analysis)
- Return top 5 most semantically similar chunks (cosine similarity)
- **Fallback**: If Pinecone returns 0 results (serverless eventual consistency lag), use locally cached chunks from the same run

---
# 🌟 Novel Methods & Design Innovations

---

## 1. LLM-Driven Domain Classification
**Why it's novel**: Most multi-agent systems use hard-coded routing ("if question contains 'price' → run pricing agent"). We use Claude to decide at runtime which agents to run, based on semantic understanding of the question. This means the system is:
- **Adaptive**: Routes correctly even for novel question types
- **Efficient**: Only runs relevant agents (2-3 agents for focused queries, all 6 for broad ones)
- **Cost-controlled**: Fewer API calls for simpler questions

---

## 2. Pydantic Few-Shot Retry Loop for Structured Extraction
**Why it's novel**: Instead of hoping Claude returns valid JSON, we:
1. Show Claude 3 domain-specific examples of correct output
2. Validate Claude's response against a Pydantic schema
3. If validation fails, feed the **exact Pydantic error back to Claude** and ask it to self-correct
4. Retry up to 3 times

This gives **structured JSON output with source attribution** even when Claude is inconsistent. Confidence scores, fact lists, and interpretation lists are all validated types.

---

## 3. Domain-Namespaced Vector Storage
**Why it's novel**: Most RAG systems use a single shared vector index. We namespace by *both* `session_id` AND `domain_tag`. This means:
- Market chunks never pollute competitive analysis retrieval
- Each domain gets exactly the content it scraped
- Queries return domain-specific context, not averaged noise

---

## 4. Dual Memory Architecture (Hot + Cold)
**Why it's novel**: Two memory systems serving different roles:
- **Pinecone** (hot): Sub-second semantic search within a session
- **Mem0** (cold): Cross-session persistent memory with semantic deduplication

Together they enable: *"I've never seen this exact question before but I remember related findings from last week"*

---

## 5. Automatic PDF Routing
**Why it's novel**: When scraping URLs from research reports, we automatically detect if a URL is a PDF and route to LlamaParse instead of Firecrawl. Standard scrapers either crash or return garbled text on PDFs. We handle this transparently, meaning analyst reports and whitepapers (the most valuable market intelligence sources) are fully parsed.

---

## 6. Graceful Degradation at Every Layer
**Why it's novel**: Every API call is wrapped in a try/except with a meaningful fallback:
- **Search fails** → continue to next query
- **Scrape fails** → return empty result, continue
- **Reddit unavailable** → skip social, continue with search+scrape only
- **Pinecone eventual consistency** → fall back to locally cached chunks
- **Domain agent fails** → store `PartialFinding(status="failed")`, continue to next domain
- **Synthesis fails** → report error in PDF, mark domain failed

The system **never fully crashes** — it always produces the best possible output from available data.

---

## 7. Base64 PDF Delivery (No Static File Server Needed)
**Why it's novel**: LangGraph dev server doesn't serve static files. Instead of setting up S3 or a file server, the PDF is encoded as a base64 data URL and embedded directly in the chat message. The user clicks a single link that opens the PDF in their browser — from a fully self-contained URL with no external dependencies.

---
# 🛠️ Technology Stack

| Layer | Technology | Purpose |
|---|---|---|
| **Orchestration** | LangGraph `StateGraph` | Sequential multi-agent state machine |
| **Chat UI** | Next.js + agent-chat-ui | Frontend with streaming `useStream` hook |
| **Primary LLM** | Claude Sonnet 4.5 (Anthropic) | Classification, synthesis, executive summary |
| **Search LLM** | Gemini Flash 2.5 (Google) | Live web search with grounding metadata |
| **Embeddings** | text-embedding-3-large (OpenAI) | 3072-dim vectors for semantic search |
| **Vector Store** | Pinecone Serverless | Session-namespaced chunk storage + retrieval |
| **Thread Memory** | Mem0 | Cross-session persistent memory with deduplication |
| **Web Scraping** | Firecrawl | HTML → clean markdown conversion |
| **PDF Parsing** | LlamaParse | OCR + layout-aware PDF extraction |
| **Social Signals** | asyncpraw (Reddit) | OAuth2 subreddit search + comments |
| **Social Signals** | HN Algolia API | Public Hacker News story search |
| **PDF Export** | WeasyPrint + Jinja2 | HTML template → PDF, base64-encoded delivery |
| **Observability** | LangFuse | LLM call tracing, token counts, latency |
| **State Schemas** | Pydantic v2 | All data models with runtime validation |
| **Async Runtime** | Python asyncio | Non-blocking I/O across all agents |

---
# 📦 Core Data Models

Everything in the system is typed with Pydantic. Here are the most important models:

## DomainFinding — Output of Every Agent
```python
class DomainFinding(BaseModel):
    domain:          str                              # "market", "competitive", etc.
    status:          Literal["complete","partial","failed"]
    summary:         str                              # 2-3 sentence synthesis
    facts:           list[Fact]                       # Grounded claims with source URLs
    interpretations: list[str]                        # Agent inferences (labelled clearly)
    confidence:      float                            # 0.0 – 1.0
    error_reason:    str | None
```

## Fact — A Single Source-Attributed Claim
```python
class Fact(BaseModel):
    claim:        str      # "CRM market projected at $83B by 2028"
    source_url:   str      # "https://www.forrester.com/..."
    retrieved_at: datetime
    confidence:   float    # 0.0 – 1.0 per claim
```

## Chunk — A Pinecone Storage Unit
```python
class Chunk(BaseModel):
    id:         str      # UUID4
    text:       str      # 512-token excerpt
    source_url: str
    domain_tag: str      # "market" / "competitive" / etc.
    agent_id:   str      # "MarketAgent" / etc.
    timestamp:  datetime
    embedding:  list[float] | None  # excluded from serialisation
```

## FinalReport — Fed to the PDF Template
```python
class FinalReport(BaseModel):
    product_name:      str
    generated_at:      datetime
    domains:           dict[str, DomainFinding]  # all 6 domains
    executive_summary: str
    failed_domains:    list[str]
```

---
# ⚡ Caching Strategy — How We Avoid Redundant Work

The system has **4 levels of caching**, from fastest to slowest:

---

## Level 1 — Python Singleton Clients (Process-Level, ~0ms)
All external API clients (Gemini, Firecrawl, Pinecone, OpenAI embedder, Anthropic) are **lazy singletons**:
```python
_bound_client: Runnable | None = None

def _get_client() -> Runnable:
    global _bound_client
    if _bound_client is None:
        _bound_client = ...  # initialise once
    return _bound_client  # reused for every call in the process
```
No repeated authentication or connection setup on every request.

---

## Level 2 — Pinecone Namespace Cache (Session-Level, ~50ms)
All chunks scraped in a session are stored in Pinecone under `namespace=session_id`. If a domain queries Pinecone and finds vectors already there from a prior run in the same session, it can recall them without re-scraping.

The `recall_or_fallback()` function in `BaseDomainAgent` also handles the case where Pinecone hasn't finished indexing yet (serverless eventual consistency) — it falls back to the locally cached chunks from the same run.

---

## Level 3 — Mem0 Domain Cache (Session+Cross-Session, ~200ms)
Before running its full pipeline, every domain agent calls:
```python
prior = await self.get_prior()          # Fetch from Mem0
if is_already_answered(prior, query):   # Heuristic check
    return DomainFinding.from_prior(prior)  # Return cached finding
```

If the agent finds a `status: complete` finding in Mem0 for this session, it **skips the entire pipeline** — no search, no scrape, no synthesis, no cost.

---

## Level 4 — Mem0 Semantic Deduplication (Write-Side, ~300ms)
When saving findings to Mem0, it doesn't just append. It **semantically matches** the new content against existing entries and updates in-place:
- "CRM market at $64B" + "CRM market at $65B" → merges into one updated entry
- Prevents memory bloat across repeated queries
- Keeps the most recent, accurate information without duplicates

---

## Cache Invalidation Strategy
The `is_already_answered()` function currently uses a **heuristic**: if any prior finding has `status: complete`, reuse it. In production, this would extend to **semantic similarity scoring** — only reuse if the current query is >90% similar to the query that produced the cached finding.

---
# 📄 What the System Produces

## The DomainFinding (per agent)
Each of the 6 agents returns a structured finding like this:

```json
{
  "domain": "market",
  "status": "complete",
  "summary": "The AI CRM market is growing at 58% CAGR, reaching $8.4B by 2028. Patent filings up 210% YoY. 47 new entrants since Q3 2024. Market at peak hype with consolidation expected 2026-2027.",
  "facts": [
    {
      "claim": "AI CRM market projected to reach $8.4B by 2028 at 58% CAGR",
      "source_url": "https://www.gartner.com/en/research/...",
      "retrieved_at": "2026-03-15T09:00:00Z",
      "confidence": 0.85
    }
  ],
  "interpretations": [
    "Category will see price compression as 47 startups compete",
    "First-mover advantage window closing; differentiate on quality not volume"
  ],
  "confidence": 0.88
}
```

## The PDF Report
All 6 domain findings are rendered into a **structured HTML template** via Jinja2, then converted to PDF by WeasyPrint. The PDF contains:

1. **Executive Summary** — 3-5 sentence cross-domain synthesis (Claude)
2. **Market & Trends** — Growth projections, CAGR, VC activity, patent signals
3. **Competitive Landscape** — Competitor matrix, G2 scores, funding rounds
4. **Win/Loss Analysis** — Churn drivers, conversion blockers, buyer quotes
5. **Pricing & Packaging** — Pricing model comparison, WTP analysis
6. **Positioning & Messaging** — Messaging map, gap analysis, competitor copy
7. **Adjacent Market Threats** — Disruption timeline, platform risks

Delivered as a **base64 data URL** — one click to download, no server required.

---
# 🎯 Summary — What Makes This System Different

| Capability | Traditional Research | Growth Intelligence Agent |
|---|---|---|
| **Time to insight** | Days to weeks | Minutes |
| **Source attribution** | Manual citation | Automatic (every fact has a URL) |
| **Data freshness** | Stale reports | Live web (Gemini grounding) |
| **Scope** | One dimension at a time | 6 simultaneous intelligence dimensions |
| **Social signals** | Ignored or manual | Reddit + HN scraped automatically |
| **Memory** | None | Cross-session Mem0 + in-session Pinecone |
| **PDF delivery** | Email attachment | Direct download from chat link |
| **Failure handling** | Fatal errors | Graceful degradation, partial results |
| **Cost control** | Fixed | Dynamic routing skips irrelevant agents |

---

## In One Sentence

> **Growth Intelligence Agent** is a sequential multi-agent AI system that automatically gathers, structures, and synthesises market intelligence from the live web — delivering a source-attributed, 6-dimensional intelligence report in minutes rather than weeks.

---

*Built on LangGraph · Claude Sonnet 4.5 · Gemini Flash 2.5 · Pinecone · Mem0 · Firecrawl · Reddit · WeasyPrint*